In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import qutip as qp

from tqdm import tqdm

In [ ]:
from tensor_networks_simulations.mps.states import MPDO, MPS
from tensor_networks_simulations.mps.algorithms import tebd_alg_choi, one_time_step_2nd_order, one_time_step_2nd_order_2approach, tebd_2nd_order_LPTN_thermal, tebd_2nd_order_LPTN_evolve_lindblad
from tensor_networks_simulations.mps.models import BondHamiltonian, H_bond_choi, HBondChoi
from tensor_networks_simulations.mps.tools import (
    correlation_one_site_mix, 
    projection_Normalization_Mix, 
    T_spin_correl_mix, 
    apply_one_site_op_mix_state, 
    expectation_value_op, 
    expectation_value, 
    apply_one_site_op_mpo
)
from tensor_networks_simulations.general_tools import ED_tools as sf
from tensor_networks_simulations.general_tools.plot_tools import figure_styling
figure_styling()

In [ ]:
L = 8
hzs = .6 * np.ones(L) 
Jzs = -0.0 * np.ones(L)
Jxs = [1.]*L
Jys = [1.]*L
hxs = mus = -0.0 * np.ones(L)
gamma= 0.0
chi_max = 32
dt=0.005
dt_list = np.array([dt,]*L)
scale = 1
tot_steps = int(4/(scale*dt))


Hb = BondHamiltonian(L, Jxs, Jys, Jzs, hxs, hzs, mus)
Hs = [Hb.nn_term(i) for i in range(L-1)]

In [ ]:
def mpo_inf_temperature(L):
    Bs_inf = MPDO().infinite_temp_state(L)
    bonds_inf = MPDO().bond_vec_inf_temp(L)
    Bs_inf, Ss_inf = MPDO().schmidt_vals_from_mps(Bs_inf, bonds_inf)
    Ms_inf, Ss_inf = MPDO().mpo_from_purified_mps(Bs_inf, Ss_inf)
    bonds_inf = bonds_inf[::2]
    bonds_inf.append(1)
    mpo_inf = MPDO(Ms_inf, Ss_inf, bonds_inf)
    return mpo_inf 



mpo_inf = mpo_inf_temperature(L)
mpo_inf = MPDO.update_mpo_with_schmidt_vals(mpo_inf)
mpo_inf = MPDO.pure_state_mpo(L)
mpo_inf = MPDO.update_mpo_with_schmidt_vals(mpo_inf)

psi = MPS.GHZ_state(L)
mpo_inf = MPDO.from_mps(psi)

phi_inf = copy.deepcopy(mpo_inf)
#phi_inf = apply_one_site_op_mpo(phi_inf, Hb.sz, int(L/2))



   
T_spin_correl_mix(phi_inf.Ms, Hb.sx, phi_inf.Ms, L, int(L/2)), projection_Normalization_Mix(mpo_inf.Ms, mpo_inf.Ms, L)


In [ ]:
mag_t = []
mx_ts =[]
mx_ts_norms = []
ts = []
t=0
deltat=0
disc_err = []
discerr = 0.
for i in tqdm(range(tot_steps)):
    mpo_inf, disc1 =  tebd_2nd_order_LPTN_evolve_lindblad(mpo_inf, gamma, Hb.sx, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    #phi_inf, disc2 =  tebd_2nd_order_LPTN(phi_inf, Hs, chi_max, L, dt_list, epsilon=10 ** (-5))
    
    t += dt
    discerr += disc1
    if np.mod(i,20)==0:
        #rho = MPDO.from_mpo(mpo)
        SzSz = T_spin_correl_mix(mpo_inf.Ms, Hb.sz, mpo_inf.Ms, L, int(L/2))
        Mz = np.sum([T_spin_correl_mix(mpo_inf.Ms, Hb.sz, mpo_inf.Ms, L, i) for i in range(L)])/L
        Mx = np.sum([T_spin_correl_mix(mpo_inf.Ms, Hb.sx, mpo_inf.Ms, L, i) for i in range(L)])/L
        print(f"""
              norm = {T_spin_correl_mix(mpo_inf.Ms, Hb.s0, mpo_inf.Ms, L, 0):.3f}, 
              <Mx> = {Mx:.3f}, Mz={Mz.real:.3f},
              t={t:.2f}, disc = {discerr:.4f},
              max_bond={max(mpo_inf.bonds)},{max(phi_inf.bonds)}
              """)
        
        mag_t.append(Mz)
        mx_ts.append(Mx)
        
        disc_err.append(discerr)
        ts.append(t)


In [ ]:
plt.plot(ts, mag_t)
plt.plot(ts, mx_ts)